In [0]:
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "auto_loader", "Schema Name")
dbutils.widgets.text("table_name", "demo_table2", "Table Name")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
table_name = dbutils.widgets.get("table_name")

full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
print(f"Selected table: {full_table_name}")

In [0]:
df = spark.read.table(full_table_name)
display(df)

In [0]:
from pyspark.sql.functions import col, when, concat_ws, lit

quality_exprs = [when(col(c).isNull(), lit(f"{c} is missing")) for c in ["id", "customer_name"]]

quality_concat = concat_ws("|", *quality_exprs)

df_quality = df.withColumn(
    "quality check",
 when(quality_concat == "", lit("no errors")).otherwise(quality_concat),
)
display(df_quality)

In [0]:
df_success = df_quality.filter(col("quality check") == "no errors")
df_failed = df_quality.filter(col("quality check") != "no errors")

display(df_success)
display(df_failed)

In [0]:
df_success = df_success.drop("_rescued_data").withColumnRenamed("quality check", "quality_check")
df_failed = df_failed.drop("_rescued_data").withColumnRenamed("quality check", "quality_check")

df_success.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.sucess_records")
df_failed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.failed_records")

print(f"Success records written to {catalog_name}.{schema_name}.sucess_records")
print(f"Failed records written to {catalog_name}.{schema_name}.failed_records")

In [0]:
%sql
select * from workspace.auto_loader.sucess_records

In [0]:
%sql
select * from workspace.auto_loader.failed_records